# CityJSON vs. triangle mesh

Left: the CityJSON object as stored (polygon rings, coloured by semantic surface type).
Right: the same object as `MeshDataset` sees it -- fan-triangulated, inner rings dropped.

The last section round-trips the mesh through the tokenizer so quantization loss is
visible separately from model error.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.visualize_cityjson import load_cityjson, cityjson_figure
from src.visualize_mesh import mesh_figure, side_by_side
from src.dataset.mesh_dataset import (
    parse_cityjson_file_to_meshes, normalize_to_unit_box, tokenize, detokenize, NUM_BINS,
)

## Pick a building

In [3]:
DATASET = ROOT / "data" / "The Hague" / "mini"
LOD = "LOD2"          # "LOD1", "LOD1_synth" or "LOD2"

path = sorted((DATASET / LOD).rglob("*.json"))[0]
cj, verts_world = load_cityjson(path)
meshes = parse_cityjson_file_to_meshes(path)

print(path.relative_to(ROOT))
print(f"{len(meshes)} objects with geometry, e.g. {list(meshes)[:3]}")

data\The Hague\mini\LOD2\Source A\Stationsbuurt_2022-04-27.json
36 objects with geometry, e.g. ['bag_0518100000312648', 'bag_0518100000308432', 'bag_0518100000256967']


In [6]:
OBJ_ID = list(meshes)[1]
v, f = meshes[OBJ_ID]

side_by_side(
    [cityjson_figure([cj["CityObjects"][OBJ_ID]], verts_world, shade=True),
     mesh_figure(v, f)],
    (f"CityJSON {LOD}: {len(list(cj['CityObjects'][OBJ_ID].get('geometry', [])))} geom",
     f"Mesh: {len(v)} verts, {len(f)} triangles -> {9 * len(f)} tokens"),
)

## Tokenizer round trip

Right-hand mesh is what the model can reproduce *at best* at this bin count --
any difference here is quantization, not the model.

In [ ]:
v_n, center, scale = normalize_to_unit_box(v)
tokens = tokenize(v_n, f, NUM_BINS)
v_rt, f_rt = detokenize(tokens, NUM_BINS)
v_rt = v_rt * scale + center

print(f"{len(tokens)} tokens, values in [{tokens.min()}, {tokens.max()}]")
print(f"verts {len(v)} -> {len(v_rt)} (bins can merge them), faces {len(f)} -> {len(f_rt)}")
print(f"max coordinate error: {scale / (NUM_BINS - 1) / 2:.3f} m (half a bin, box {scale:.1f} m)")

side_by_side([mesh_figure(v, f), mesh_figure(v_rt, f_rt, color="#eda100")],
             ("Original mesh", f"Round-tripped, {NUM_BINS} bins"))

## The actual training item: LOD1 condition vs. LOD2 target

Both are normalized in the **LOD1** bounding box, which is the only frame available
at sampling time -- so they are directly comparable here.

In [ ]:
from src.dataset.mesh_dataset import MeshDataset

ds = MeshDataset(DATASET, max_files=1)          # one tile only, keeps this quick
k = 0
(v1, f1), (v2, f2) = ds.mesh_pair(k)
item = ds[k]

print(f"{ds.ids[k]}: cond {len(item['cond'])} tokens, tgt {len(item['tgt'])} tokens "
      f"(incl. BOS/EOS), dataset max_seq_len {ds.max_seq_len}")

side_by_side([mesh_figure(v1, f1, color="#898781"), mesh_figure(v2, f2)],
             (f"LOD1 condition, {len(f1)} tris", f"LOD2 target, {len(f2)} tris"))